## Crawler

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from config import TIME_OUT


class Crawler:

    def crawl_website(self, url: str) -> str:
        try:
            response = requests.get(url, timeout=TIME_OUT, headers={
                                    'User-Agent': 'rag-chatbot/5.0'})
            response.raise_for_status()
            return response.text
        except Exception:
            return ""

    def extract_links(self, base_url, html_content):
        soup = BeautifulSoup(html_content, 'html.parser')
        links = set()
        for link in soup.find_all('a', href=True):
            absolute_link = urljoin(base_url, link['href'])
            parsed_link = urlparse(absolute_link)
            if parsed_link.scheme in ['http', 'https']:
                links.add(absolute_link)
        return links

    def cleaner(self, html_content):
        soup = BeautifulSoup(html_content, "html.parser")

        for tag in soup(["nav", "footer", "header", "script", "style"]):
            tag.decompose()

        text = soup.get_text(separator=" ")
        return " ".join(text.split())

    def crawl(self, url, max_pages=10):
        visited = set()
        to_visit = [url]
        all_content = []

        while to_visit and len(visited) < max_pages:
            current_url = to_visit.pop(0)
            if current_url in visited:
                continue

            print(f"Crawling page {len(visited)+1}: {current_url}")
            content = self.crawl_website(url=current_url)

            if content:

                visited.add(current_url)

                hyperlinks = self.extract_links(
                    base_url=current_url, html_content=content)
                content = self.cleaner(content)
                all_content.append(content)
                for link in hyperlinks:
                    if link not in visited:
                        to_visit.append(link)

        return all_content


In [2]:
crawler = Crawler()
data = crawler.crawl("https://en.wikipedia.org/wiki/Artificial_intelligence")
data


Crawling page 1: https://en.wikipedia.org/wiki/Artificial_intelligence
Crawling page 2: https://en.wikipedia.org/wiki/Artificial_intelligence#cite_ref-35
Crawling page 3: https://en.wikipedia.org/wiki/Artificial_intelligence#cite_ref-347
Crawling page 4: https://en.wikipedia.org/wiki/Artificial_intelligence#cite_note-91
Crawling page 5: https://en.wikipedia.org/wiki/Artificial_intelligence#cite_note-345
Crawling page 6: https://min.wikipedia.org/wiki/Kacerdasan_buatan
Crawling page 7: https://en.wikipedia.org/wiki/Artificial_intelligence#cite_ref-FOOTNOTERussellNorvig202122_396-0
Crawling page 8: https://en.wikipedia.org/wiki/Artificial_intelligence#cite_ref-FOOTNOTESearle19801_456-0
Crawling page 9: https://en.wikipedia.org/wiki/Stanford.edu
Crawling page 10: https://en.wikipedia.org/wiki/FCC


['Artificial intelligence - Wikipedia Jump to content From Wikipedia, the free encyclopedia Intelligence of machines "AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation) . Part of a series on Artificial intelligence (AI) Major goals Artificial general intelligence Intelligent agent Recursive self-improvement Planning Computer vision General game playing Knowledge representation Natural language processing Robotics AI safety Approaches Machine learning Symbolic Deep learning Bayesian networks Evolutionary algorithms Hybrid intelligent systems Systems integration Open-source Applications Bioinformatics Deepfake Earth sciences Finance Generative AI Art Audio Music Government Healthcare Mental health Industry Software development Translation Military Physics Projects Philosophy AI alignment Artificial consciousness The bitter lesson Chinese room Friendly AI Ethics Existential risk Turing test Uncanny valley Human–AI interaction History T

In [3]:
display(data)
    

['Artificial intelligence - Wikipedia Jump to content From Wikipedia, the free encyclopedia Intelligence of machines "AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation) . Part of a series on Artificial intelligence (AI) Major goals Artificial general intelligence Intelligent agent Recursive self-improvement Planning Computer vision General game playing Knowledge representation Natural language processing Robotics AI safety Approaches Machine learning Symbolic Deep learning Bayesian networks Evolutionary algorithms Hybrid intelligent systems Systems integration Open-source Applications Bioinformatics Deepfake Earth sciences Finance Generative AI Art Audio Music Government Healthcare Mental health Industry Software development Translation Military Physics Projects Philosophy AI alignment Artificial consciousness The bitter lesson Chinese room Friendly AI Ethics Existential risk Turing test Uncanny valley Human–AI interaction History T

## Embedding

In [4]:
# from sentence_transformers import SentenceTransformer
import numpy as np
from google.genai import types
from google import genai
import faiss
import pickle
import os
from config import FAISS_INDEX_PATH, CHUNKS_PATH, MIN_PARA_LENGTH, MAX_PARA_LENGTH, DATA_FOLDER, EMBEDDING_DIMENSION, BATCH_SIZE, API_KEY, EMBEDDING_MODEL


class Chunk_generator:
    def chunk_text(self, texts: list):
        chunks = []
        for text in texts:
            paras = text.split("\n")
            for para in paras:
                if len(para) > MIN_PARA_LENGTH:
                    # further split long paragraphs
                    if len(para) > MAX_PARA_LENGTH:
                        for i in range(0, len(para), 1500):
                            chunks.append(para[i: i + 1500])
                    else:
                        chunks.append(para)
        return chunks


class Embedder:

    # def generate_embedding(self, chunks):
    #     embedder = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")
    #     embeddings = embedder.encode(chunks, show_progress_bar=True)
    #     return embeddings
    def __init__(self):
        self.client = genai.Client(api_key=API_KEY)

    def generate_embedding(self, chunks, batch_size=BATCH_SIZE):
        all_embeddings = []

        # Process the list in chunks of batch_size
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i: i + batch_size]

            config = types.EmbedContentConfig(
                output_dimensionality=EMBEDDING_DIMENSION,
                task_type="RETRIEVAL_DOCUMENT"  # Recommended for RAG storage
            )

            result = self.client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=batch,
                config=config
            )

            # Extract numerical values from the response
            for embedding in result.embeddings:
                all_embeddings.append(embedding.values)

        return np.array(all_embeddings, dtype=np.float32)


class FaissStore:
    def save_index(self, embeddings, chunks):
        os.makedirs(DATA_FOLDER, exist_ok=True)

        dim = embeddings.shape[1]
        index = faiss.IndexFlatL2(dim)
        index.add(embeddings)

        faiss.write_index(index, FAISS_INDEX_PATH)

        with open(CHUNKS_PATH, "wb") as f:
            pickle.dump(chunks, f)

    def load_index(self):
        if not os.path.exists(FAISS_INDEX_PATH):
            return None, None

        index = faiss.read_index(FAISS_INDEX_PATH)

        with open(CHUNKS_PATH, "rb") as f:
            chunks = pickle.load(f)

        return index, chunks


In [5]:
chunk= Chunk_generator()
embedder= Embedder()
store= FaissStore()

chunk_store= chunk.chunk_text(data)
embedding_store= embedder.generate_embedding(chunk_store)
store.save_index(embeddings=embedding_store, chunks=chunk_store)
index,stored_chunk= store.load_index()


## Q|A

In [6]:
import os
from google import genai
from google.genai import types
from config import TOP_K, API_KEY, GEMINI_MODEL, FAILED_CONTEXT_RETRIEVAL
from embedding import Chunk_generator, Embedder, FaissStore
from crawler import Crawler


class QA:
    def __init__(self):
        self.store = FaissStore()
        self.embedder = Embedder()
        self.client = genai.Client(api_key=API_KEY)
        self.crawler = Crawler()
        self.chunk_generator = Chunk_generator()

    def retrieve_chunks(self, query):

        index, chunks = self.store.load_index()
        if index is None:
            return []

        query_vec = self.embedder.generate_embedding([query])
        _, indices = index.search(query_vec, TOP_K)

        return [chunks[i] for i in indices[0]]

    def ask_gemini(self, context, question, previous_conversation=[]):
        prompt = f"""
            You must answer ONLY using the context below.
            If the answer is not present, respond exactly with:
            "The answer is not available on the provided website."

            Context:
            {context}

            Previous_conversation:
            {previous_conversation}
            
            Question:
            {question}
            
        """
        response = self.client.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
        )
        return response.text.strip()

    def answer_question(self, question, previous_conversation=[]):
        retrieved = self.retrieve_chunks(question)

        if not retrieved:
            return FAILED_CONTEXT_RETRIEVAL

        context = "\n\n".join(retrieved)
        return self.ask_gemini(context, question, previous_conversation)

    def indexing(self, url, max_pages=10):
        data = self.crawler.crawl(url, max_pages)
        chunks = self.chunk_generator.chunk_text(data)
        embeddings = self.embedder.generate_embedding(chunks)
        self.store.save_index(embeddings=embeddings, chunks=chunks)


In [8]:
qa=QA()
qa.indexing(url='https://takeuforward.org/dsa/strivers-a2z-sheet-learn-dsa-a-to-z')


Crawling page 1: https://takeuforward.org/dsa/strivers-a2z-sheet-learn-dsa-a-to-z
Crawling page 2: https://takeuforward.org/
Crawling page 3: https://takeuforward.org/home
Crawling page 4: https://takeuforward.org/plus/home
Crawling page 5: https://takeuforward.org/blogs/two-pointers
Crawling page 6: https://takeuforward.org/linked-list/top-linkedlist-interview-questions-structured-path-with-video-solutions
Crawling page 7: https://takeuforward.org/data-structure/two-sum-check-if-a-pair-with-given-sum-exists-in-array?mode=track&sheet=blind-75
Crawling page 8: https://takeuforward.org/blogs/greedy
Crawling page 9: https://takeuforward.org/blogs/js
Crawling page 10: https://takeuforward.org/dynamic-programming/striver-dp-series-dynamic-programming-problems


In [9]:
qa.answer_question("what is takeuforward")

'takeuforward - Best Coding Tutorials for Free'